# Proyecto RAG · Ley 19.628 (reformada por la Ley 21.719)

Sistema RAG completo sobre la ley de protección de datos personales de Chile.

**Idea del proyecto.** Construir un RAG que responda preguntas sobre la ley usando *solo* los artículos pertinentes y **citando la fuente**, y mejorar la calidad con re-ranking. Recorre las Rúbricas **R1 a R6**: chunking (R1), vector store en Qdrant (R2), RAG con fuentes (R3), re-ranking (R4), umbral (R5) y mini-evaluación (R6).

**Documento fuente.** Usamos el **texto refundido de la Ley 19.628** con las modificaciones de la Ley 21.719 ya aplicadas (versión vigente a 2026). Elegimos esta versión consolidada —en vez del texto modificatorio de la 21.719— porque presenta los artículos limpios y numerados, lo que hace que las **citas sean correctas**.

https://www.bcn.cl/leychile/navegar?idNorma=141599&idVersion=2026-12-01

## 0 · Configuración

Las credenciales (OpenAI y Qdrant) ya están en **Colab → Secrets** (Fase 0) y se usarán desde la Fase 3.

In [1]:
# Librerías de chunking (las mismas de tu Notebook_02 del curso)
!pip -q install langchain-text-splitters langchain-core

### Cargar el documento fuente (portable para todo el grupo)

La celda busca `Ley_19628_refundida_21719.txt` en la sesión de Colab (o junto al notebook, si corre local); si no lo encuentra, abre el diálogo para **subirlo desde el PC**. Así el notebook corre en el Colab de **cualquier integrante**, sin depender de rutas de un Drive personal. Si prefieres usar Drive, basta dejar el archivo en la raíz de `MyDrive` y montar el Drive antes.

In [ ]:
# Carga portable: sesión de Colab / carpeta local / subida manual / Drive (opcional)
from pathlib import Path

NOMBRE = "Ley_19628_refundida_21719.txt"
candidatas = [
    Path(NOMBRE),                              # subido a la sesión de Colab o junto al notebook
    Path("/content/drive/MyDrive") / NOMBRE,   # raíz de tu Drive (si ya lo montaste)
]

ruta = next((p for p in candidatas if p.exists()), None)
if ruta is None:
    from google.colab import files             # última opción: subirlo desde el PC
    subidos = files.upload()
    ruta = Path(next(iter(subidos)))

ley = ruta.read_text(encoding="utf-8")
print(f"{ruta.name}: {len(ley):,} caracteres cargados")

## Fase 2 · Chunking  (Rúbrica 1)

El sistema no puede leer toda la ley en cada pregunta; la cortamos en *fichas* (chunks). Al preguntar, busca las pocas fichas más parecidas y responde con esas.

**Nuestra decisión y por qué.** La ley ya viene dividida en **artículos**, y cada artículo es una norma autocontenida (una ficha). Por eso:

1. **Cortamos por artículo** (estrategia *por estructura*): mantiene cada norma entera.
2. **Sub-cortamos solo los artículos largos** (más de 1.500 caracteres) con `RecursiveCharacterTextSplitter`, en trozos de unos 1.200 con 150 de solape.
3. **Anteponemos el encabezado del artículo a cada fragmento** (*contextual chunking*): así ningún trozo pierde su contexto (p. ej., la lista de infracciones graves conserva la palabra "graves").
4. **Guardamos metadata** (`source`, `articulo`) en cada chunk: es lo que luego permite **citar la fuente** (Fase 4).

*Medido sobre este documento:* 87 artículos → 192 chunks.

In [3]:
# Chunking: por artículo + sub-cortar largos + ANTEPONER encabezado a cada fragmento + metadata
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

patron = re.compile(
    r'(?=Artículo\s+(?:primero|segundo|tercero|cuarto|quinto|sexto|séptimo|octavo|noveno|décimo|\d+\s*[°º]?)'
    r'(?:\s+(?:bis|ter|quáter|quinquies|sexies))?)'
)
bloques = [b.strip() for b in patron.split(ley) if b.strip()]

def etiqueta_articulo(b):
    m = re.match(r'(Artículo\s+[^\n\.\-]{1,20})', b)
    return m.group(1).strip() if m else "Preámbulo"

def encabezado(b):   # p. ej. "Artículo 34 ter.- Infracciones graves."
    m = re.match(r'(Artículo\s+[^\n]{1,45}?\.-\s*[^\.\n]{1,60}?\.)', b)
    return m.group(1).strip() if m else etiqueta_articulo(b)

sub = RecursiveCharacterTextSplitter(
    chunk_size=1200, chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)
LIMITE = 1500

docs = []
for b in bloques:
    art = etiqueta_articulo(b)
    enc = encabezado(b)
    piezas = [b] if len(b) <= LIMITE else sub.split_text(b)
    for j, p in enumerate(piezas):
        # cada fragmento lleva el encabezado de su artículo: el contexto no se pierde
        texto = p if p.startswith("Artículo") else f"[{enc}] {p}"
        docs.append(Document(
            page_content=texto,
            metadata={"source": "Ley 19.628 (ref. 21.719)", "articulo": art, "parte": f"{j+1}/{len(piezas)}"},
        ))

print(f"{len(bloques)} artículos -> {len(docs)} chunks (con encabezado en cada fragmento)")

87 artículos -> 192 chunks (con encabezado en cada fragmento)


In [4]:
# Inspeccionar el resultado
largos = [len(d.page_content) for d in docs]
print("Tamaño (caracteres): min", min(largos),
      "· mediana", sorted(largos)[len(largos)//2],
      "· máx", max(largos))

# Un chunk de ejemplo, con su metadata
docs[10].metadata, docs[10].page_content[:200]

Tamaño (caracteres): min 76 · mediana 945 · máx 1468


({'source': 'Ley 19.628 (ref. 21.719)',
  'articulo': 'Artículo 2°',
  'parte': '8/8'},
 '[Artículo 2°.- Definiciones.] prevención, los responsables de datos que los hayan\nadoptado y las sanciones que se hayan impuesto a los\nresponsables de datos que hayan infringido la ley.')

### Comparación de configuraciones (PAra justificación de Fase 2)

Para justificar la elección, comparamos nuestra configuración contra dos cortes "a ciegas" por tamaño fijo. Medimos tres cosas: cuántas normas quedan **enteras** en un solo chunk, cuántos chunks **mezclan** dos o más artículos, y si se conserva la **metadata** del artículo (necesaria para citar).

In [5]:
# Comparación: nuestra config vs. dos cortes fijos "a ciegas"
import re, statistics
norm = lambda s: re.sub(r'\s+', ' ', s).strip()
MARK = re.compile(
    r'Artículo\s+(?:primero|segundo|tercero|cuarto|quinto|sexto|séptimo|octavo|noveno|décimo|\d+\s*[°º]?)'
    r'(?:\s+(?:bis|ter|quáter|quinquies|sexies))?'
)

def fixed(t, size, ov):
    out, i = [], 0
    while i < len(t):
        out.append(t[i:i+size]); i += size - ov
    return out

nuestra = [d.page_content for d in docs]
configs = {
    "Nuestra (por art. 1200/150)": nuestra,
    "Fixed 300 (a ciegas)":  fixed(ley, 300, 30),
    "Fixed 2000 (a ciegas)": fixed(ley, 2000, 200),
}

# Muestra de normas completas (artículos que caben en un chunk nuestro)
muestra = [norm(b) for b in bloques if 350 <= len(b) <= 1400]

def integridad(chunks):   # % de normas que quedan ENTERAS en un solo chunk
    chs = [norm(c) for c in chunks]
    ok = sum(1 for a in muestra if any(a in c for c in chs))
    return round(100 * ok / len(muestra))

def mezclan(chunks):      # nº de chunks que contienen 2+ artículos distintos
    return sum(1 for c in chunks if len(MARK.findall(c)) >= 2)

print(f"{'Config':<28}{'chunks':>7}{'medcar':>8}{'enteras':>9}{'mezclan':>9}{'metadata':>10}")
for n, chs in configs.items():
    med = int(statistics.median([len(c) for c in chs]))
    meta = "Si" if n.startswith("Nuestra") else "No"
    print(f"{n:<28}{len(chs):>7}{med:>8}{str(integridad(chs))+'%':>9}{mezclan(chs):>9}{meta:>10}")

Config                       chunks  medcar  enteras  mezclan  metadata
Nuestra (por art. 1200/150)     192     943     100%        0        Si
Fixed 300 (a ciegas)            568     300       0%        2        No
Fixed 2000 (a ciegas)            86    2000      42%       20        No


### Justificación del chunking — Rúbrica 1

Elegimos trocear la Ley 19.628 (refundida) **por artículo**, con sub-corte recursivo de 1.200/150 caracteres solo para los artículos largos y **anteponiendo el encabezado del artículo a cada fragmento**, guardando el número de artículo como metadata (87 artículos → 192 chunks). Comparamos contra dos cortes a ciegas por tamaño fijo (ver tabla):

- Con **fixed-300**, el **0 %** de las normas queda entero: cada artículo se parte en fragmentos y una definición se corta a la mitad.
- Con **fixed-2000**, **20 chunks mezclan** dos o más artículos y solo el **42 %** de las normas queda entero, perdiendo precisión.
- Además, ninguna configuración fija conserva el artículo de origen, por lo que **no permite citar la fuente**.

Nuestra configuración mantiene el **100 %** de las normas enteras, no mezcla artículos (**0**) y conserva la metadata para citar. Anteponer el encabezado evita, además, que fragmentos como la lista de infracciones graves pierdan su contexto. Por eso la elegimos.

In [6]:
# Librerías adicionales: embeddings + Qdrant
!pip -q install langchain-openai langchain-qdrant qdrant-client openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 18.0 MB/s eta 0:00:00


In [8]:
# Puente de credenciales (Secrets → entorno) y constantes
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["QDRANT_URL"]      = userdata.get("QDRANT_URL")
os.environ["QDRANT_API_KEY"]  = userdata.get("QDRANT_API_KEY")

EMBED_MODEL = "text-embedding-3-large"   # modelo de embeddings del curso
EMBED_DIMS  = 256                          # con 256 dimensiones siguiendo la configuración del Notebook 03 del curso; el modelo permite reducir dimensiones sin perder casi calidad, y 256 es más liviano de almacenar y buscar.
COLLECTION  = "ley_21719_rag"             # NUESTRA colección (no la del curso)

In [9]:
# Modelo de embeddings + chequeo de dimensión
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model=EMBED_MODEL, dimensions=EMBED_DIMS)
print("dimensiones del embedding:", len(embeddings.embed_query("prueba")))  # debe ser 256

dimensiones del embedding: 256


In [10]:
# Crear la colección en Qdrant y subir los chunks (Rúbrica 2)
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])

# 256 dims + distancia coseno. recreate = idempotente: re-ejecutar no duplica.
client.recreate_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBED_DIMS, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(client=client, collection_name=COLLECTION, embedding=embeddings)
vector_store.add_documents(docs)   # aquí se calculan los embeddings y se suben

print("Vectores en la colección:", client.count(collection_name=COLLECTION).count)  # debe ser 192

/tmp/ipykernel_4574/1288338681.py:9: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Vectores en la colección: 192


In [11]:
# Un registro de ejemplo (id + payload + vector) — Rúbrica 2
puntos, _ = client.scroll(collection_name=COLLECTION, limit=1, with_payload=True, with_vectors=True)
p = puntos[0]
print("id:", p.id)
print("payload:", p.payload)                         # el texto + metadata (source, articulo)
print("vector (primeros 5 de 256):", [round(x, 4) for x in p.vector[:5]])

id: 0057d43a-c194-4a4e-a229-626a9b4788bf
payload: {'page_content': 'Artículo 22.- Comunicación o cesión de datos por un\nórgano público. Los órganos públicos están facultados\npara comunicar o ceder datos personales específicos, o todo\no parte de sus bases de datos o conjuntos de datos, a otros\nórganos públicos, siempre que la comunicación o cesión\nde los datos resulte necesaria para el cumplimiento de sus\nfunciones legales y ambos órganos actúen dentro del\námbito de sus competencias. La comunicación o cesión de\nlos datos se debe realizar para un tratamiento específico y\nel órgano público receptor no los podrá utilizar para\notros fines.\n    Asimismo, se podrá comunicar o ceder datos o bases de\ndatos personales entre organismos públicos, exclusivamente\ncuando ellos se requieran para un tratamiento que tenga por\nfinalidad otorgar beneficios al titular, evitar duplicidad\nde trámites para los ciudadanos o reiteración de\nrequerimientos de información o documentos para los mism

## Fase 4 · RAG básico con fuentes  (Rúbrica 3)

Conectamos las piezas: la pregunta se convierte en vector, se recuperan los pasajes más cercanos (top-k), se arma un contexto con ellos y un modelo generador responde **usando solo ese contexto y citando el artículo**. Si la respuesta no está en el contexto, lo dice (clave para no inventar — la pregunta trampa). El prompt está endurecido para ser conciso y no ofrecer extrapolaciones.

In [12]:
# RAG básico: pregunta → recuperar top-k → responder citando el artículo (Rúbrica 3)
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

GEN_MODEL = "gpt-5.4-mini"   # modelo generador del curso
TOP_K = 5

def formatear(docs):
    # Une los pasajes con su artículo, para poder citar la fuente
    return "\n\n".join(f"[{d.metadata.get('articulo','?')}] {d.page_content}" for d in docs)

# Prompt de respuesta endurecido: conciso, sin ofertas, sin salir del contexto
answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Eres un asistente legal. Responde la PREGUNTA usando EXCLUSIVAMENTE el CONTEXTO. "
     "Reglas: (1) si la respuesta no está en el contexto, responde solo "
     "'No se encuentra en el documento.' y nada más; "
     "(2) no agregues información fuera del contexto ni ofrezcas resúmenes o extrapolaciones; "
     "(3) sé conciso y responde solo lo preguntado; "
     "(4) cita el artículo entre [corchetes] tal como aparece en el contexto."),
    ("human", "CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"),
])

generador = ChatOpenAI(model=GEN_MODEL, reasoning_effort="low")
rag = answer_prompt | generador | StrOutputParser()

def responder(pregunta):
    docs = vector_store.similarity_search(pregunta, k=TOP_K)
    return rag.invoke({"contexto": formatear(docs), "pregunta": pregunta})

In [13]:
# Demo con las preguntas de oro (incluida la trampa) — Rúbrica 3
preguntas = [
    "¿Qué es un dato personal sensible?",
    "¿Cómo debe ser el permiso para que usen mis datos?",
    "¿Qué organismo crea la ley y para qué?",
    "¿Qué infracción es grave y qué multa arriesga?",
    "¿Qué dice la ley sobre la inteligencia artificial?",   # trampa: no está
]
for q in preguntas:
    print("P:", q)
    print("R:", responder(q))
    print("-" * 80)

P: ¿Qué es un dato personal sensible?
R: [Artículo 2°] Aquellos datos personales que se refieren a las características físicas o morales de las personas o a hechos o circunstancias de su vida privada o intimidad, que revelen el origen étnico o racial, la afiliación política, sindical o gremial, la situación socioeconómica, las convicciones ideológicas o filosóficas, las creencias religiosas, los datos relativos a la salud, al perfil biológico humano, los datos biométricos, y la información relativa a la vida sexual, a la orientación sexual y a la identidad de género de una persona natural.
--------------------------------------------------------------------------------
P: ¿Cómo debe ser el permiso para que usen mis datos?
R: El consentimiento debe ser **libre, informado, específico en cuanto a su finalidad o finalidades, previo e inequívoco**, mediante **declaración verbal, escrita, por medio electrónico equivalente o un acto afirmativo** que muestre con claridad la voluntad del titula

## Fase 5 · Re-ranking + umbral  (Rúbricas 4 y 5)

El RAG básico recupera por similitud (rápido pero aproximado). El re-ranking añade un filtro fino: recuperamos **amplio (top-20)**, un **LLM juez** puntúa la relevancia real de cada pasaje de 0 a 1 (R4), y nos quedamos solo con los que superan un **umbral** (R5). El prompt del juez está adaptado a textos legales: no penaliza definiciones ni listas, porque en una ley *son* la respuesta.

In [14]:
# Re-ranking (R4) + umbral (R5)
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

FAST_MODEL = "gpt-5.4-nano"   # modelo barato para puntuar (del curso)
TOP_K_WIDE = 20               # recuperamos amplio
THRESHOLD  = 0.55             # umbral de relevancia (ajustable)

class Relevancia(BaseModel):
    score: float = Field(description="Relevancia de 0.0 (irrelevante) a 1.0 (responde directamente).")

rerank_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un evaluador de relevancia para un sistema RAG sobre una ley. "
               "¿Qué tan relevante es el PASAJE para responder la PREGUNTA? "
               "Devuelve un número de 0.0 a 1.0. En textos legales, las definiciones, "
               "listas y enumeraciones a menudo SON la respuesta: no las penalices. "
               "Puntúa alto si el pasaje contiene la información que responde la pregunta."),
    ("human", "PREGUNTA: {pregunta}\n\nPASAJE: {pasaje}"),
])
scorer = rerank_prompt | ChatOpenAI(model=FAST_MODEL).with_structured_output(Relevancia)

def responder_rerank(pregunta):
    candidatos = vector_store.similarity_search(pregunta, k=TOP_K_WIDE)              # 1. amplio
    entradas = [{"pregunta": pregunta, "pasaje": d.page_content} for d in candidatos]
    scores = [r.score for r in scorer.batch(entradas)]                              # 2. juez (R4)
    rankeados = sorted(zip(candidatos, scores), key=lambda x: x[1], reverse=True)
    sobrevivientes = [d for d, s in rankeados if s >= THRESHOLD]                     # 3. umbral (R5)
    if not sobrevivientes:
        return "No se encuentra en el documento."
    return rag.invoke({"contexto": formatear(sobrevivientes), "pregunta": pregunta}) # 4. responder

In [15]:
# Ver el re-ranking en acción en la pregunta 4, y comparar con el básico
import pandas as pd
p = "¿Qué infracción es grave y qué multa arriesga?"

cands = vector_store.similarity_search(p, k=TOP_K_WIDE)
scores = [r.score for r in scorer.batch([{"pregunta": p, "pasaje": d.page_content} for d in cands])]
ranked = sorted(zip(cands, scores), key=lambda x: x[1], reverse=True)

print(pd.DataFrame([{"score": round(s, 2), "pasa": "✔" if s >= THRESHOLD else "✘",
                     "articulo": d.metadata["articulo"], "texto": d.page_content[:55] + "..."}
                    for d, s in ranked]).head(8).to_string(index=False))

print("\nBÁSICO:   ", responder(p))
print("\nRE-RANKED:", responder_rerank(p))

 score pasa           articulo                                                       texto
  0.95    ✔        Artículo 35 Artículo 35.- Sanciones. Las sanciones a las\ninfraccion...
  0.85    ✔ Artículo 34 quáter  [Artículo 34 quáter.- Infracciones gravísimas.] a) Efec...
  0.82    ✔        Artículo 35  [Artículo 35.- Sanciones.] En cada caso, la Agencia señ...
  0.78    ✔    Artículo 34 ter  [Artículo 34 ter.- Infracciones graves.] estudios o inv...
  0.78    ✔        Artículo 44  [Artículo 44] la determinación de la sanción se deberán...
  0.74    ✔ Artículo 34 quáter  [Artículo 34 quáter.- Infracciones gravísimas.] penales...
  0.72    ✔    Artículo 34 ter  [Artículo 34 ter.- Infracciones graves.] a) Tratar los ...
  0.70    ✔    Artículo 34 ter  [Artículo 34 ter.- Infracciones graves.] solicitudes fu...

BÁSICO:    No se encuentra en el documento.

RE-RANKED: Las infracciones graves se sancionan con multa de hasta 10.000 unidades tributarias mensuales [Artículo 35]. Por ejemplo, tr

## Fase 6 · Mini-evaluación  (Rúbrica 6)

Comparamos de forma sistemática **básico vs re-ranked** sobre las 5 preguntas de oro, cada una con su respuesta esperada. Un **juez LLM calibrado** (reglas estrictas + ejemplo) puntúa cada respuesta, y promediamos varias corridas para lidiar con la variación propia de los LLM.

In [16]:
# Preguntas de oro CON su respuesta esperada
gold = [
    ("¿Qué es un dato personal sensible?",
     "Datos de características físicas/morales o de la vida privada que revelan origen étnico, afiliación política/sindical, salud, datos biométricos, vida/orientación sexual, identidad de género. (Art. 2°)"),
    ("¿Cómo debe ser el permiso para que usen mis datos?",
     "El consentimiento debe ser libre, informado, específico, previo e inequívoco. (Art. 12)"),
    ("¿Qué organismo crea la ley y para qué?",
     "La Agencia de Protección de Datos Personales, para fiscalizar y regular el tratamiento de datos. (Art. 30 bis)"),
    ("¿Qué infracción es grave y qué multa arriesga?",
     "Ej.: tratar datos sin consentimiento es infracción grave (Art. 34 ter); multa de hasta 10.000 UTM (Art. 35)."),
    ("¿Qué dice la ley sobre la inteligencia artificial?",
     "La ley no regula la IA de forma expresa; lo más cercano son las decisiones automatizadas (Art. 8° bis)."),
]

In [17]:
# Fase 6 — Mini-eval automática con juez estricto y calibrado (R6)
import pandas as pd
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

class Nota(BaseModel):
    razon: str = Field(description="Justificación en una frase (razona antes de puntuar).")
    contiene_hecho_clave: bool = Field(description="¿La respuesta contiene el hecho central de la esperada?")
    cita_correcta: bool = Field(description="¿Cita el artículo correcto?")
    score: float = Field(description="0.0 si falta el hecho o dice que no lo encuentra; 0.5 si está el hecho pero la cita falla; 1.0 si ambos.")

juez_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Eres un evaluador ESTRICTO de un sistema RAG legal. Compara la RESPUESTA con la ESPERADA.\n"
     "- Si la respuesta dice 'no se encuentra' o no contiene el hecho central de la esperada -> 0.0\n"
     "- Si contiene el hecho central pero no cita el artículo correcto -> 0.5\n"
     "- Si contiene el hecho central y cita el artículo correcto -> 1.0\n"
     "Excepción: si la esperada dice que el tema NO está en la ley y la respuesta también, -> 1.0.\n"
     "Ejemplo: esperada 'multa 10.000 UTM (Art. 35)'; respuesta 'No se encuentra' -> 0.0."),
    ("human", "PREGUNTA: {pregunta}\n\nESPERADA: {esperada}\n\nRESPUESTA: {respuesta}"),
])
juez = juez_prompt | ChatOpenAI(model=GEN_MODEL, reasoning_effort="high").with_structured_output(Nota)

N_RUNS = 2
tabla = []
for q, esperada in gold:
    sb = [juez.invoke({"pregunta": q, "esperada": esperada, "respuesta": responder(q)}).score for _ in range(N_RUNS)]
    sr = [juez.invoke({"pregunta": q, "esperada": esperada, "respuesta": responder_rerank(q)}).score for _ in range(N_RUNS)]
    tabla.append({"pregunta": q[:30] + "...", "básico": round(sum(sb)/N_RUNS, 2), "re-ranked": round(sum(sr)/N_RUNS, 2)})

df = pd.DataFrame(tabla)
df["mejora"] = df.apply(lambda r: "sube" if r["re-ranked"] > r["básico"] else ("igual" if r["re-ranked"] == r["básico"] else "baja"), axis=1)
print(df.to_string(index=False))
print(f"\nPromedio  básico: {df['básico'].mean():.2f}   re-ranked: {df['re-ranked'].mean():.2f}")

                         pregunta  básico  re-ranked mejora
¿Qué es un dato personal sensi...     1.0        1.0  igual
¿Cómo debe ser el permiso para...     1.0        1.0  igual
¿Qué organismo crea la ley y p...     1.0        0.5   baja
¿Qué infracción es grave y qué...     0.0        0.5   sube
¿Qué dice la ley sobre la inte...     1.0        1.0  igual

Promedio  básico: 0.80   re-ranked: 0.80


### Conclusión de la mini-eval — Rúbrica 6

Comparamos el RAG básico contra el RAG con re-ranking sobre las 5 preguntas de oro, usando un juez LLM calibrado (reglas estrictas + respuesta esperada de referencia) y promediando 2 corridas. Como el pipeline usa modelos de lenguaje —que no son deterministas—, las notas exactas y el promedio varían levemente entre ejecuciones, por lo que reportamos la **tendencia** y no cada decimal. El resultado **estable en todas las corridas** es la pregunta multi-salto (infracciones graves): el básico no logra recuperar el listado del Artículo 34 ter y responde "No se encuentra", mientras que el re-ranking sí lo recupera y responde, obteniendo una nota **igual o superior** a la del básico. En las preguntas directas ambos pipelines rinden alto y de forma pareja, y las diferencias puntuales en otras preguntas se explican por esa misma variabilidad. **Conclusión:** el re-ranking aporta de forma consistente donde la similitud simple deja fuera el pasaje correcto (las preguntas que exigen combinar o rescatar contenido específico), sin degradar el desempeño en las preguntas fáciles.

# Bonus UI con gradio

In [18]:
# Bonus · Interfaz con Gradio
!pip -q install gradio

In [22]:
# Bonus · UI: escribe una pregunta y ve la respuesta con su cita (usa tu pipeline de la Fase 5)
import gradio as gr

def consultar(pregunta):
    if not pregunta.strip():
        return "Escribe una pregunta sobre la ley."
    return responder_rerank(pregunta)   # ← tu RAG con re-ranking (Fase 5)

demo = gr.Interface(
    fn=consultar,
    inputs=gr.Textbox(label="Tu pregunta sobre la Ley 19.628",
                      placeholder="¿Qué es un dato personal sensible?"),
    outputs=gr.Markdown(label="Respuesta (con cita del artículo)"),
    title="Asistente RAG · Ley de Protección de Datos",
    description="Responde usando solo la ley y citando el artículo. Si no está, lo dice.",
    examples=[
        "¿Qué es un dato personal sensible?",
        "¿Qué infracción es grave y qué multa arriesga?",
        "¿Qué dice la ley sobre la inteligencia artificial?",
    ],
     flagging_mode="never",
)
demo.launch(share=True)   # 'share=True' te da un enlace público para la demo

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0fe0eada349814307b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Bonus · Multi-query / HyDE (capa de consulta)

Dos técnicas que atacan el mismo problema desde el lado de la **pregunta**: el usuario no habla "en legalés". Si pregunta *"¿puedo pedir que borren mis datos?"*, la ley dice *"derecho de supresión"* — y la similitud coseno puede no anclar.

- **Multi-query:** un LLM genera 3 reformulaciones con vocabulario legal, se busca con las 4 versiones (original + 3) y se fusionan los rankings con **RRF** (Reciprocal Rank Fusion: suma por *posición* en cada ranking, no por score, así no hay que calibrar escalas).
- **HyDE (Hypothetical Document Embeddings):** un LLM redacta un *artículo hipotético* que respondería la pregunta, y se busca con el embedding de **ese texto**. Comparar documento-contra-documento ancla mejor que pregunta-contra-documento.

Ambas reutilizan el índice de Qdrant y la cadena `rag` existentes: solo cambia **qué se busca**, no cómo se responde.

In [ ]:
# Bonus · Multi-query: reformular con vocabulario legal y fusionar rankings con RRF
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

class Variantes(BaseModel):
    variantes: list[str] = Field(description="Tres reformulaciones de la pregunta, distintas entre sí.")

mq_prompt = ChatPromptTemplate.from_messages([
    ("system", "Reformula la PREGUNTA de un usuario para buscar en la Ley 19.628 de protección "
               "de datos personales (Chile). Genera 3 variantes con el vocabulario legal-formal "
               "que usaría la ley (p. ej. 'borrar' -> 'supresión'; 'permiso' -> 'consentimiento'; "
               "'empresa' -> 'responsable de datos')."),
    ("human", "PREGUNTA: {pregunta}"),
])
generar_variantes = mq_prompt | ChatOpenAI(model=FAST_MODEL).with_structured_output(Variantes)

def buscar_multiquery(pregunta, k=TOP_K, por_variante=4, verbose=True):
    """Busca con la pregunta original + 3 reformulaciones y fusiona por RRF (posición, no score)."""
    consultas = [pregunta] + generar_variantes.invoke({"pregunta": pregunta}).variantes
    if verbose:
        for c in consultas[1:]:
            print("  · variante:", c)
    puntaje, doc_por_clave = {}, {}
    for q in consultas:
        for pos, d in enumerate(vector_store.similarity_search(q, k=por_variante)):
            clave = (d.metadata["articulo"], d.metadata.get("parte"))
            puntaje[clave] = puntaje.get(clave, 0) + 1 / (60 + pos + 1)   # RRF
            doc_por_clave[clave] = d
    mejores = sorted(puntaje, key=puntaje.get, reverse=True)[:k]
    return [doc_por_clave[c] for c in mejores]

def responder_multiquery(pregunta):
    docs_rec = buscar_multiquery(pregunta)
    return rag.invoke({"contexto": formatear(docs_rec), "pregunta": pregunta})

# Demo: pregunta coloquial que NO usa el vocabulario de la ley
p = "¿Puedo pedir que borren mis datos de una empresa?"
print("BÁSICO     :", responder(p))
print()
print("MULTI-QUERY:", responder_multiquery(p))

In [ ]:
# Bonus · HyDE: buscar con un "artículo hipotético" en vez de con la pregunta
from langchain_core.output_parsers import StrOutputParser

hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "Escribe un párrafo breve (3 a 4 líneas) redactado como un artículo de la "
               "Ley 19.628 que respondería la PREGUNTA. Usa lenguaje legal chileno formal. "
               "No importa si los detalles son inventados: el texto se usa SOLO para buscar, "
               "nunca se muestra al usuario."),
    ("human", "PREGUNTA: {pregunta}"),
])
articulo_hipotetico = hyde_prompt | ChatOpenAI(model=FAST_MODEL) | StrOutputParser()

def responder_hyde(pregunta, k=TOP_K):
    """Genera el texto hipotético y busca con SU embedding (doc-contra-doc ancla mejor)."""
    hipotetico = articulo_hipotetico.invoke({"pregunta": pregunta})
    print("  · hipotético:", hipotetico.replace("\n", " ")[:150], "…")
    docs_rec = vector_store.similarity_search(hipotetico, k=k)
    return rag.invoke({"contexto": formatear(docs_rec), "pregunta": pregunta})

print("HYDE:", responder_hyde("¿Puedo pedir que borren mis datos de una empresa?"))

# Bonus · RAGAS — métricas estándar de evaluación

La mini-eval de la Fase 6 usa un juez "casero". **RAGAS** agrega métricas *reconocidas y comparables* de la industria, cada una entre 0 y 1:

| Métrica | Pregunta que responde | Qué parte del sistema evalúa |
|---|---|---|
| **faithfulness** | ¿Cada afirmación de la respuesta se apoya en el contexto recuperado? | Generación (mide alucinación) |
| **answer_relevancy** | ¿La respuesta responde *esa* pregunta y no otra cosa? | Generación |
| **context_precision** | De lo recuperado, ¿cuánto era realmente útil y quedó bien rankeado? | Retrieval + re-ranking |
| **context_recall** | ¿El contexto alcanzó para cubrir la respuesta esperada (gold)? | Retrieval |

Evaluamos el pipeline **re-ranked** sobre las mismas 5 preguntas de oro, armando el dataset que RAGAS exige: `question`, `answer`, `contexts` (los pasajes que pasaron el umbral) y `ground_truth` (la respuesta esperada).

**Notas de lectura.** (1) En la pregunta trampa (IA) las métricas de contexto pueden salir bajas *por diseño*: la referencia dice que el tema no está en la ley — se comenta, no se oculta. (2) Cada métrica hace varias llamadas al LLM por pregunta: la celda demora unos minutos y tiene costo. (3) Si la API de RAGAS cambiara con una versión nueva, fijar la versión (p. ej. `pip install "ragas==0.2.*"`).

In [ ]:
# Bonus · RAGAS (1/2): instalar y preparar
!pip -q install "ragas==0.4.3"

# Shim de compatibilidad: ragas 0.4.3 todavía importa dos clases VertexAI legacy que
# langchain-community >= 0.4 (en desuso) ya no trae. Solo se usan para chequeos de tipo,
# así que si faltan se registran clases vacías con el mismo nombre.
import sys, types
try:
    from langchain_community.chat_models.vertexai import ChatVertexAI  # noqa: F401
except Exception:
    for _n in ("langchain_community", "langchain_community.chat_models",
               "langchain_community.chat_models.vertexai", "langchain_community.llms"):
        if _n not in sys.modules:
            try:
                __import__(_n)
            except Exception:
                sys.modules[_n] = types.ModuleType(_n)
    class _VertexDummy:   # nunca se usa: solo existe para el isinstance interno de ragas
        pass
    sys.modules["langchain_community.chat_models.vertexai"].ChatVertexAI = _VertexDummy
    sys.modules["langchain_community.llms"].VertexAI = _VertexDummy

import ragas
print("ragas", ragas.__version__, "listo")

In [ ]:
# Bonus · RAGAS (2/2): evaluar el pipeline re-ranked con métricas estándar
from ragas import evaluate, EvaluationDataset
from ragas.dataset_schema import SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from langchain_openai import ChatOpenAI

llm_ragas = LangchainLLMWrapper(ChatOpenAI(model=GEN_MODEL))   # juez que usa RAGAS
emb_ragas = LangchainEmbeddingsWrapper(embeddings)             # para answer_relevancy

def contextos_rerank(pregunta):
    """Los pasajes que realmente usó el pipeline re-ranked (los que pasan el umbral)."""
    candidatos = vector_store.similarity_search(pregunta, k=TOP_K_WIDE)
    scores = [r.score for r in scorer.batch([{"pregunta": pregunta, "pasaje": d.page_content}
                                             for d in candidatos])]
    rankeados = sorted(zip(candidatos, scores), key=lambda x: x[1], reverse=True)
    return [d.page_content for d, s in rankeados if s >= THRESHOLD]

# Dataset en el formato canónico de RAGAS: pregunta, respuesta, contextos usados y referencia
muestras = []
for pregunta, esperada in gold:
    muestras.append(SingleTurnSample(
        user_input=pregunta,
        response=responder_rerank(pregunta),
        retrieved_contexts=contextos_rerank(pregunta) or ["(ningún pasaje superó el umbral)"],
        reference=esperada,
    ))

resultado = evaluate(EvaluationDataset(samples=muestras),
                     metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
                     llm=llm_ragas, embeddings=emb_ragas)
print(resultado)          # promedio por métrica
resultado.to_pandas()     # detalle por pregunta